# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### 1.The Content Performance Curve
This presents health score(label) by content-age buckets as if it were lifecycle; content peaks, then declines,then rebounds. But the paper's methodology supports a 90-day rolling window with no evidence of tracking individual pages over their full life. This means the age buckets almost certainly compare different pages published at different times, not the same pages aging. The lifecycle framing implies causation that a cross-sectional comparison can't actually support. Pages published a year ago could simply differ systematically from pages published last week, independent of age itself.

### 2. Click Capture By Position Tier
This presents a weighted CTR as its label, which is a solid pick, but it was stated that we 'the CTR makes page-one refinements a more reliable source of incremental clicks than spreading effort evenly across pages with little existing visibility'. But this begs the question, is it the CTR that makes a page get the top position or is it the quality of the page that gets it to the top position, hence bringing the CTR numbers. What I imply is that it is wiser to update based on quality potential of the particular page instead of discarding the other pages that are not at top positions.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\hamto\OneDrive\Desktop\Flyrank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df = df[df['search_volume'].notna() & df['cpc'].notna()]

In [3]:
df['trend_pct_flipped'] = -df['trend_pct']
lower = df['trend_pct_flipped'].quantile(0.01)
upper = df['trend_pct_flipped'].quantile(0.99)

df['trend_pct_dampened'] = df['trend_pct_flipped'].clip(lower, upper)

In [4]:
df['refresh_score'] = df['trend_pct_dampened'] * df['search_volume'] * df['cpc']

In [5]:
df = df[df['trend_pct'].notna()]
threshold = df['refresh_score'].quantile(0.90)

df['is_priority'] = (df['refresh_score'] >= threshold).astype(int)
df['is_priority'].value_counts()

is_priority
0    22758
1     2529
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

train_df_random, test_df_random = train_test_split(df, test_size=0.2, random_state=42)

In [7]:
overlap = set(train_df_random['client_id']) & set(test_df_random['client_id'])
print(len(overlap))  # should be 0

30


In [8]:
drop_cols = [
    'client_id', 'content_id', 'is_priority', 'trend_pct', 'trend_pct_flipped',
    'trend_pct_dampened', 'refresh_score', 'cpc', 'search_volume',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'trend_direction',
    'provider_used', 'model_used', 'age_tier', 'impression_tier', 'position_tier',
    'freshness_tier', 'char_count_tier', 'word_count_tier',
    ]  # your full list above
X_train_random = train_df_random.drop(columns=drop_cols)
y_train_random = train_df_random['refresh_score']

X_test_random = test_df_random.drop(columns=drop_cols)
y_test_random = test_df_random['refresh_score']

X_train_random.dtypes

competition               float64
competition_level             str
content_type                  str
main_intent                   str
word_count                float64
char_count                float64
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
content_age_days            int64
age_tier_order              int64
days_since_last_update      int64
ctr                       float64
avg_position              float64
engagement_rate           float64
scroll_rate               float64
ai_traffic_pct            float64
dtype: object

In [9]:
train_df_random.groupby('content_type')['word_count'].apply(lambda x: x.isna().mean())

content_type
comparison article    0.000000
keyword article       0.294545
Name: word_count, dtype: float64

In [10]:
for col in ['word_count', 'char_count']:
    X_train_random[f'{col}_missing'] = X_train_random[col].isna().astype(int)
    X_test_random[f'{col}_missing'] = X_test_random[col].isna().astype(int)

    median_val = X_train_random[col].median()  # compute from TRAIN only
    X_train_random[col] = X_train_random[col].fillna(median_val)
    X_test_random[col] = X_test_random[col].fillna(median_val)  # apply train's median to test too

In [11]:
# numeric — fill with train median
for col in ['competition', 'scroll_rate']:
    median_val = X_train_random[col].median()
    X_train_random[col] = X_train_random[col].fillna(median_val)
    X_test_random[col] = X_test_random[col].fillna(median_val)

# categorical — explicit "missing" category, not a guess
for col in ['competition_level', 'main_intent']:
    X_train_random[col] = X_train_random[col].fillna('missing')
    X_test_random[col] = X_test_random[col].fillna('missing')

In [12]:
X_train_encoded = pd.get_dummies(X_train_random)
X_test_encoded = pd.get_dummies(X_test_random)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(X_train_encoded.shape)
print(X_test_encoded.shape)

(20229, 34)
(5058, 34)


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_encoded, y_train_random)
rf_preds = rf.predict(X_test_encoded)

In [14]:
def precision_at_k(preds, actual, k):
    results = pd.DataFrame({'pred': preds, 'actual': actual})
    results_sorted = results.sort_values('pred', ascending=False)
    return (results_sorted.head(k)['actual'] == 1).mean()

In [15]:
precision_at_k(rf_preds, test_df_random['is_priority'].values, 20)

np.float64(0.45)

At k = 20, both grouped and random splits happened to score identically (0.45), which could be misread as 'the split doesn't matter'. But checking across multiple k values shows that the two splits genuinely diverges at other K's. The K=20 being the same is a small coincidence and is not an evidence that client leakage has no effect.This is a reminder that a single precision@kK value at one K is a noisy signal on its own. 

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [17]:
import pandas as pd 
importances = pd.Series(rf.feature_importances_, index=X_train_encoded.columns)
importances.sort_values(ascending=False).head(10)

avg_position          0.352574
content_age_days      0.086525
sessions_90d          0.067401
impressions_90d       0.062475
pageviews_90d         0.058338
days_with_sessions    0.053172
competition           0.039283
word_count            0.030740
users_90d             0.029910
scroll_rate           0.029581
dtype: float64

## REMARKS
The top 10 feature importances confirm my week 5 findings: avg_position dominates at 255, and the remaining importance is spread across decline/engagement signals(content_age_days, sessions_90d, impressions_90d, e.t.c.). Critically, no feature in the model carries a value signal; nothing analogous to cpc or sear_volume exists in a non-leaky form. This is not a leakage problem, but a structural gap: the model has no way to distinguish a high-value declining page from a low-value one, explaining why it over-ranks low-cpc pages that were correctly identified as wrong picks in week 5.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Random Forest's top-20 predictions matched is_priority at a rate of 0.45 (9 out of 20), roughly 4x the base rate of 0.113. This is a measured improvement over random ranking, not evidence if reliable prediction,. The model still mis-ranked 11 of its 20 picks. Error analysis suggests this stems from a structural gap: the model has no value signal(excluding cpc and search_volume because of leakage), so it cannot distinguish high-value declining pages from low-value ones. Adding a non- leaky value proxy is a directional next step, not a guaranteed fix.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.